# Color-invariant saree design retrieval

Identify a saree by the motif on its surface, not by the palette it was printed in.
The gallery is a set of known photographs. A query should rank the photograph of the
same design above photographs of other designs.

This notebook does not train a classifier. The four folder names in the Kaggle export
(Banarasi, Bandhani, Ikat, Pichwai) are weave families. Many different motifs sit inside
each folder, so those names are not design ids and they are never used as a loss target.

## Approach note (487 characters)

> Frozen DINOv2 ViT-B/14 (timm vit_base_patch14_dinov2, the dinov2_vitb14 weights) extracts 768-d embeddings. Nothing is trained and ArcFace is not used: the four weave folders are not motif IDs. Inference embeds RGB, grayscale, and Canny edges, L2-normalizes each vector, and averages them so color cannot dominate geometry. Ranking is cosine similarity plus k-reciprocal re-ranking. Positives are Roboflow .rf. copies of one source photo. Rank-1, mAP, and verification use that protocol.

## How a score should be read

Roboflow stored three salt-and-pepper copies of many source photos, named
`original.rf.<hash>.jpg`. Those copies are the only positive pairs this dataset
supports. A high Rank-1 on that protocol means the embedding is stable to that
augmentation. It does not, by itself, prove that two colorways of one motif outrank
two different motifs. The hue-shift probe later in the notebook is the color check
that can be run without motif labels: the query is recolored and must still find
its original copy.


## 0. Runtime

Run this on a GPU runtime (Colab: Runtime, Change runtime type, GPU). The setup cell
stops if PyTorch cannot see a CUDA device, so a CPU-only wheel cannot silently do
the forward passes. On the RTX 4050 laptop, install a CUDA build of PyTorch first.
Do not pip-install `torch` inside Colab: the GPU runtime already has a CUDA build,
and replacing it with a CPU wheel is the usual way to "lose" the GPU.


In [ ]:
import importlib
import subprocess
import sys

def _need(module: str, package: str) -> None:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

# torch and torchvision stay as the runtime provided them.
_need("timm", "timm>=1.0.9")
_need("cv2", "opencv-python-headless>=4.8")
_need("sklearn", "scikit-learn>=1.3")
_need("tqdm", "tqdm>=4.66")
print("dependencies ready")


In [ ]:
import os

import torch

print("torch", torch.__version__)
print("cuda compiled:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "PyTorch is running on CPU. Stop here.\n"
        "Colab: Runtime -> Change runtime type -> GPU, then rerun from the top.\n"
        "RTX 4050 laptop: this environment has a CPU wheel. Install a CUDA wheel, then rerun:\n"
        "  pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124"
    )

DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
on_colab = "COLAB_RELEASE_TAG" in os.environ
print(f"active GPU: {GPU_NAME} ({GPU_GB:.1f} GB)")
if "4050" in GPU_NAME:
    print("RTX 4050 is the device that will run the forward passes.")
elif on_colab:
    print("Colab GPU runtime is active. Forward passes will use it rather than the laptop CPU.")
else:
    print("Forward passes will use the CUDA device printed above.")

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")


## 1. Retrieval helpers

The next cell is the tested implementation of grouping, Rank-1, mAP, and
k-reciprocal re-ranking (`deeplure_core.py`). Short filenames such as `image19.jpeg`
were reused in every weave folder, so the group id is the folder name plus the
source stem. That folder name only stops unrelated files from sharing an id.


In [ ]:
"""Zero-shot color-invariant saree retrieval.

DINOv2 ViT-B/14 is a frozen feature extractor. Weave-folder names are never
used as training labels. This module holds the pieces that do not need a GPU
so they can be tested on their own: grouping, metrics, and k-reciprocal
re-ranking.
"""

from __future__ import annotations

from collections import defaultdict
from pathlib import Path

import numpy as np

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


def source_group(path: Path) -> str:
    """Identity of the photo before Roboflow's .rf.<hash> copy suffix.

    `stem.rf.abc123.jpg` and `stem.rf.def456.jpg` are copies of one source
    image, so they share a group. Files without `.rf.` are their own group.
    """
    name = path.name
    marker = ".rf."
    if marker in name:
        return name.split(marker)[0]
    return path.stem


def discover_roboflow(root: Path) -> list[dict]:
    """Index train/valid/test class folders.

    The group id is the weave folder plus the source stem. Short names such as
    `image19.jpeg` were reused inside every class folder, so the stem alone
    would glue unrelated photographs together. The weave folder is only a
    namespace that keeps those files apart. It is not a class the model learns.
    """
    root = Path(root)
    records = []
    if not root.exists():
        return records
    for split_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        for class_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
            for path in sorted(class_dir.iterdir()):
                if path.suffix.lower() not in IMAGE_EXTS or not path.is_file():
                    continue
                stem = source_group(path)
                records.append(
                    {
                        "path": path.resolve(),
                        "group": f"{class_dir.name}/{stem}",
                        "weave": class_dir.name,
                        "split": split_dir.name,
                    }
                )
    return records


def discover_flat(root: Path) -> list[dict]:
    """Index an unlabeled folder. Each file is its own group."""
    root = Path(root)
    if not root.exists():
        return []
    records = []
    for path in sorted(root.iterdir()):
        if path.suffix.lower() not in IMAGE_EXTS or not path.is_file():
            continue
        records.append(
            {
                "path": path.resolve(),
                "group": path.stem,
                "weave": "",
                "split": "unlabeled",
            }
        )
    return records


def audit_groups(records: list[dict]) -> dict:
    """Summarize whether .rf. copies stayed in one folder and one weave name."""
    by_group = defaultdict(list)
    for rec in records:
        by_group[rec["group"]].append(rec)
    sizes = [len(v) for v in by_group.values()]
    split_across_folders = 0
    mixed_weave = 0
    for members in by_group.values():
        if len({m["split"] for m in members}) > 1:
            split_across_folders += 1
        weaves = {m["weave"] for m in members if m["weave"]}
        if len(weaves) > 1:
            mixed_weave += 1
    usable = sum(1 for n in sizes if n >= 2)
    return {
        "images": len(records),
        "groups": len(by_group),
        "groups_with_a_positive": usable,
        "max_copies": max(sizes) if sizes else 0,
        "groups_split_across_folders": split_across_folders,
        "groups_with_mixed_weave_names": mixed_weave,
    }


def query_gallery_split(records: list[dict]) -> tuple[list[int], list[int]]:
    """One gallery reference per source photo, remaining copies are queries.

    The first filename in a group is the reference. Groups with a single file
    have no positive and are left out. Query and gallery indices are disjoint.
    """
    by_group = defaultdict(list)
    for index, rec in enumerate(records):
        by_group[rec["group"]].append(index)

    queries: list[int] = []
    gallery: list[int] = []
    for group in sorted(by_group):
        members = sorted(by_group[group], key=lambda i: records[i]["path"].name)
        if len(members) < 2:
            continue
        gallery.append(members[0])
        queries.extend(members[1:])
    return queries, gallery


def rank1_and_map(scores: np.ndarray, query_groups: list[str], gallery_groups: list[str]) -> dict:
    """Rank-1 and mAP from a query-by-gallery score matrix (higher is closer).

    A gallery item is relevant when its group equals the query group. Average
    precision uses the precision at each relevant rank.
    """
    scores = np.asarray(scores, dtype=np.float64)
    if scores.ndim != 2:
        raise ValueError("scores must be a query-by-gallery matrix")
    n_query, n_gallery = scores.shape
    if n_query != len(query_groups) or n_gallery != len(gallery_groups):
        raise ValueError("group lists must match the score matrix shape")

    order = np.argsort(-scores, axis=1, kind="mergesort")
    gallery_groups = list(gallery_groups)
    hits = []
    average_precisions = []
    reciprocal_ranks = []

    for row, ranked in enumerate(order):
        target = query_groups[row]
        n_rel = sum(group == target for group in gallery_groups)
        if n_rel == 0:
            continue
        hits.append(gallery_groups[int(ranked[0])] == target)
        found = 0
        precision_sum = 0.0
        first_rank = None
        for rank, col in enumerate(ranked, start=1):
            if gallery_groups[int(col)] != target:
                continue
            found += 1
            precision_sum += found / rank
            if first_rank is None:
                first_rank = rank
            if found == n_rel:
                break
        average_precisions.append(precision_sum / n_rel)
        reciprocal_ranks.append(1.0 / first_rank if first_rank else 0.0)

    n = len(average_precisions)
    return {
        "rank1": float(np.mean(hits)) if n else float("nan"),
        "mAP": float(np.mean(average_precisions)) if n else float("nan"),
        "mrr": float(np.mean(reciprocal_ranks)) if n else float("nan"),
        "n_queries": n,
        "n_gallery": n_gallery,
    }


def k_reciprocal_distance(similarity: np.ndarray, k1: int = 20, k2: int = 6, lam: float = 0.3) -> np.ndarray:
    """Pairwise distance after k-reciprocal re-ranking.

    Implements Zhong, Zheng, Cao, and Li, "Re-ranking Person Re-identification
    with k-reciprocal Encoding", CVPR 2017. `similarity` is cosine similarity.
    The returned matrix is a distance: smaller means more similar. Diagonal is 0.

    Final distance = (1 - lam) * cosine_distance + lam * jaccard_distance.
    """
    similarity = np.asarray(similarity, dtype=np.float64)
    if similarity.ndim != 2 or similarity.shape[0] != similarity.shape[1]:
        raise ValueError("similarity must be square")
    n = similarity.shape[0]
    if n == 0:
        return similarity.copy()

    dist = np.clip(1.0 - similarity, 0.0, None)
    np.fill_diagonal(dist, 0.0)
    # Rank neighbors from the raw cosine distance. A copy is kept because the
    # expansion step must not see the later blended distance.
    initial_rank = np.argsort(dist, axis=1, kind="mergesort")
    k1 = int(max(1, min(k1, n - 1)))
    k2 = int(max(1, min(k2, n)))
    half = max(1, int(round(k1 / 2.0)))

    encoding = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        forward = initial_rank[i, : k1 + 1]
        backward = initial_rank[forward, : k1 + 1]
        reciprocal_slots = np.where(backward == i)[0]
        reciprocal = forward[reciprocal_slots]

        expanded = reciprocal
        for candidate in reciprocal:
            cand_forward = initial_rank[candidate, : half + 1]
            cand_backward = initial_rank[cand_forward, : half + 1]
            cand_slots = np.where(cand_backward == candidate)[0]
            cand_reciprocal = cand_forward[cand_slots]
            overlap = np.intersect1d(cand_reciprocal, reciprocal, assume_unique=False)
            if cand_reciprocal.size > 0 and overlap.size > (2.0 / 3.0) * cand_reciprocal.size:
                expanded = np.append(expanded, cand_reciprocal)
        expanded = np.unique(expanded)
        weights = np.exp(-dist[i, expanded])
        encoding[i, expanded] = weights / np.sum(weights)

    if k2 > 1:
        smoothed = np.empty_like(encoding)
        for i in range(n):
            smoothed[i] = encoding[initial_rank[i, :k2]].mean(axis=0)
        encoding = smoothed

    # Inverted lists keep the Jaccard loop on the non-zero support only.
    inverted = [np.flatnonzero(encoding[:, col]) for col in range(n)]
    jaccard = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        support = np.flatnonzero(encoding[i])
        if support.size == 0:
            jaccard[i] = 1.0
            jaccard[i, i] = 0.0
            continue
        overlap_mass = np.zeros(n, dtype=np.float64)
        for col in support:
            rows = inverted[col]
            overlap_mass[rows] += np.minimum(encoding[i, col], encoding[rows, col])
        # Both rows are L1-normalized, so sum(max) = 2 - sum(min).
        jaccard[i] = 1.0 - overlap_mass / np.clip(2.0 - overlap_mass, 1e-12, None)
        jaccard[i, i] = 0.0

    blended = (1.0 - lam) * dist + lam * jaccard
    np.fill_diagonal(blended, 0.0)
    return blended


def reranked_query_scores(
    embeddings: np.ndarray,
    query_index: list[int],
    gallery_index: list[int],
    k1: int = 20,
    k2: int = 6,
    lam: float = 0.3,
) -> np.ndarray:
    """Cosine k-reciprocal scores for queries against a disjoint gallery.

    Re-ranking is computed on the union of query and gallery images, which is
    the closed-set setting used at evaluation. Higher scores are closer.
    """
    if set(query_index) & set(gallery_index):
        raise ValueError("query and gallery indices overlap")
    union = list(gallery_index) + list(query_index)
    position = {image_index: row for row, image_index in enumerate(union)}
    features = np.asarray(embeddings, dtype=np.float64)[union]
    norms = np.linalg.norm(features, axis=1, keepdims=True)
    features = features / np.clip(norms, 1e-12, None)
    similarity = features @ features.T
    distance = k_reciprocal_distance(similarity, k1=k1, k2=k2, lam=lam)
    rows = [position[i] for i in query_index]
    cols = [position[i] for i in gallery_index]
    return -distance[np.ix_(rows, cols)]


def reranked_from_similarity(
    similarity: np.ndarray,
    query_index: list[int],
    gallery_index: list[int],
    k1: int = 20,
    k2: int = 6,
    lam: float = 0.3,
) -> np.ndarray:
    """Re-rank a full pairwise cosine matrix and return query-gallery scores.

    `similarity[i, j]` is the cosine of dataset row i with dataset row j.
    Higher returned scores are closer.
    """
    if set(query_index) & set(gallery_index):
        raise ValueError("query and gallery indices overlap")
    union = list(gallery_index) + list(query_index)
    position = {image_index: row for row, image_index in enumerate(union)}
    block = np.asarray(similarity, dtype=np.float64)[np.ix_(union, union)]
    distance = k_reciprocal_distance(block, k1=k1, k2=k2, lam=lam)
    rows = [position[i] for i in query_index]
    cols = [position[i] for i in gallery_index]
    return -distance[np.ix_(rows, cols)]


def cosine_query_scores(embeddings: np.ndarray, query_index: list[int], gallery_index: list[int]) -> np.ndarray:
    features = np.asarray(embeddings, dtype=np.float64)
    norms = np.linalg.norm(features, axis=1, keepdims=True)
    features = features / np.clip(norms, 1e-12, None)
    return features[query_index] @ features[gallery_index].T


def topk_indices(scores: np.ndarray, k: int) -> tuple[np.ndarray, np.ndarray]:
    """Return gallery indices and scores of the top-k matches per query."""
    k = int(min(k, scores.shape[1]))
    if k <= 0:
        empty = np.zeros((scores.shape[0], 0), dtype=np.int64)
        return empty, empty.astype(np.float64)
    chosen = np.argpartition(-scores, kth=k - 1, axis=1)[:, :k]
    row = np.arange(scores.shape[0])[:, None]
    order = np.argsort(-scores[row, chosen], axis=1, kind="mergesort")
    cols = chosen[row, order]
    return cols, scores[row, cols]


## 2. Data paths and model settings

The notebook looks for the Kaggle export (a folder that contains `train`, `valid`,
and `test`) and for the unlabeled handloom folder. Images are not in the git
repository. On Colab, put them on Drive and uncomment the mount, or set
`ROBOFLOW_ROOT` and `HANDLOOM_ROOT`.


In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
print("Uncomment the Drive lines above on Colab if the images live on Drive.")


In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

# DINOv2 ViT-B/14 was trained at 518, which is 37 patches of 14.
# Drop this to 224 (16 patches) if a small GPU runs out of memory at batch size 1.
IMG_SIZE = 518
K1, K2, LAMBDA = 20, 6, 0.3
HUE_SHIFT = 90  # OpenCV hue is 0-179, so 90 flips the palette about 180 degrees.
FORCE_RECOMPUTE = False
SEED = 0
TOPK = 5

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
VIEWS = ("rgb", "gray", "canny")

if IMG_SIZE % 14 != 0:
    raise ValueError("IMG_SIZE must be a multiple of the DINOv2 patch size, 14")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
CACHE_PATH = OUTPUT_DIR / f"dinov2_views_{IMG_SIZE}.npz"

# 518px ViT-B is comfortable at batch 4 on a 16 GB Colab GPU and tight on a 6 GB 4050.
BATCH_SIZE = 8 if GPU_GB >= 14 else 4
print(f"image size {IMG_SIZE}, starting batch size {BATCH_SIZE}")


def resolve_roots():
    candidates = []
    if os.environ.get("ROBOFLOW_ROOT"):
        candidates.append((Path(os.environ["ROBOFLOW_ROOT"]), Path(os.environ.get("HANDLOOM_ROOT", ""))))
    candidates.extend([
        (Path(r"C:\Users\dhruv\Downloads\archive"),
         Path(r"C:\Users\dhruv\Downloads\handloom_sarees-20261005T103013Z-1-001\handloom_sarees")),
        (Path("/content/drive/MyDrive/DeepLure/archive"),
         Path("/content/drive/MyDrive/DeepLure/handloom_sarees")),
        (Path("/content/data/archive"), Path("/content/data/handloom_sarees")),
        (Path("/content/archive"), Path("/content/handloom_sarees")),
    ])
    for roboflow, handloom in candidates:
        if (roboflow / "train").is_dir():
            return roboflow, handloom if handloom.is_dir() else None
    searched = "\n".join(str(r) for r, _ in candidates)
    raise FileNotFoundError(
        "Could not find the Kaggle export (a folder containing train/valid/test).\n"
        "Set ROBOFLOW_ROOT, or place it at one of:\n" + searched
    )


ROBOFLOW_ROOT, HANDLOOM_ROOT = resolve_roots()
print("roboflow:", ROBOFLOW_ROOT)
print("handloom:", HANDLOOM_ROOT if HANDLOOM_ROOT else "(not found, qualitative section will be skipped)")

APPROACH = 'Frozen DINOv2 ViT-B/14 (timm vit_base_patch14_dinov2, the dinov2_vitb14 weights) extracts 768-d embeddings. Nothing is trained and ArcFace is not used: the four weave folders are not motif IDs. Inference embeds RGB, grayscale, and Canny edges, L2-normalizes each vector, and averages them so color cannot dominate geometry. Ranking is cosine similarity plus k-reciprocal re-ranking. Positives are Roboflow .rf. copies of one source photo. Rank-1, mAP, and verification use that protocol.'


In [ ]:
records = discover_roboflow(ROBOFLOW_ROOT)
summary = audit_groups(records)
query_index, gallery_index = query_gallery_split(records)
print("images", summary["images"])
print("source photos", summary["groups"])
print("photos with at least one Roboflow copy", summary["groups_with_a_positive"])
print("max copies of one photo", summary["max_copies"])
print("groups split across train/valid/test", summary["groups_split_across_folders"])
print("groups whose copies carry different weave names", summary["groups_with_mixed_weave_names"])
print("queries", len(query_index), "gallery", len(gallery_index))
if summary["groups_with_mixed_weave_names"] or summary["max_copies"] > 3:
    print("WARNING: grouping does not look like the expected sets of 3 copies. Check filenames before trusting Rank-1.")

paths = [rec["path"] for rec in records]
query_groups = [records[i]["group"] for i in query_index]
gallery_groups = [records[i]["group"] for i in gallery_index]


## 3. Color-invariant views

Each forward pass sees one of three renderings of the same resized photo:

- RGB, the original pixels
- grayscale, replicated to three channels
- Canny edges (thresholds 100 and 200, fixed ahead of time, not tuned on the queries)

The three 768-d embeddings are each L2-normalized, averaged, and L2-normalized again.
That average is the descriptor used for matching. Grayscale and edges still respond
to motif geometry after the palette changes. Averaging them with RGB keeps the
descriptor from being only an edge sketch.


In [ ]:
def hue_rotate(image: Image.Image, shift: int) -> Image.Image:
    array = np.asarray(image.convert("RGB"))
    hsv = cv2.cvtColor(array, cv2.COLOR_RGB2HSV)
    hsv[:, :, 0] = (hsv[:, :, 0].astype(np.int16) + int(shift)) % 180
    return Image.fromarray(cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB))


def load_rgb(path: Path, image_size: int, hue_shift: int = 0) -> Image.Image:
    try:
        with Image.open(path) as handle:
            image = handle.convert("RGB")
    except Exception as exc:
        print("unreadable", path, exc)
        image = Image.new("RGB", (image_size, image_size), (0, 0, 0))
    image = image.resize((image_size, image_size), Image.Resampling.BICUBIC)
    if hue_shift:
        image = hue_rotate(image, hue_shift)
    return image


def render_view(image: Image.Image, view: str) -> Image.Image:
    if view == "rgb":
        return image
    if view == "gray":
        return ImageOps.grayscale(image).convert("RGB")
    if view == "canny":
        gray = np.asarray(ImageOps.grayscale(image))
        edges = cv2.Canny(gray, 100, 200)
        return Image.fromarray(cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB))
    raise KeyError(view)


_TO_TENSOR = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class SareeViewDataset(Dataset):
    # One rendered view per path. hue_shift is applied before the view is made.

    def __init__(self, file_paths, view: str, image_size: int, hue_shift: int = 0):
        self.file_paths = list(file_paths)
        self.view = view
        self.image_size = image_size
        self.hue_shift = hue_shift

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, index):
        rgb = load_rgb(self.file_paths[index], self.image_size, self.hue_shift)
        return _TO_TENSOR(render_view(rgb, self.view))


# Four source photos, one rendering of each view, so the transform is visible before any score.
preview_groups = []
seen = set()
for record in records:
    if record["group"] in seen:
        continue
    seen.add(record["group"])
    preview_groups.append(record)
    if len(preview_groups) == 4:
        break

figure, axes = plt.subplots(len(preview_groups), 4, figsize=(10, 2.4 * len(preview_groups)))
for row, record in enumerate(preview_groups):
    rgb = load_rgb(record["path"], 256, 0)
    panels = [rgb, render_view(rgb, "gray"), render_view(rgb, "canny"), hue_rotate(rgb, HUE_SHIFT)]
    titles = ["RGB", "grayscale", "Canny", f"hue +{HUE_SHIFT}"]
    for col, (panel, title) in enumerate(zip(panels, titles)):
        axes[row, col].imshow(panel)
        axes[row, col].set_title(title if row == 0 else "")
        axes[row, col].axis("off")
    axes[row, 0].set_ylabel(record["weave"], rotation=90, size=8)
figure.suptitle("Inputs the embedder actually sees, plus the hue-shift probe")
figure.tight_layout()
figure.savefig(OUTPUT_DIR / "views.png", dpi=120)
plt.show()


## 4. Frozen DINOv2 ViT-B/14

`num_classes=0` returns the 768-d pooled embedding and drops the classification
head. Weights stay frozen. The checkpoint is the public DINOv2 ViT-B/14 release
distributed through timm (`vit_base_patch14_dinov2`), which is the `dinov2_vitb14`
model.


In [ ]:
import timm

model = timm.create_model("vit_base_patch14_dinov2", pretrained=True, num_classes=0)
model.eval().to(DEVICE)
model.requires_grad_(False)

cfg = model.pretrained_cfg
print("architecture:", cfg.get("architecture"))
print("checkpoint:", cfg.get("url") or cfg.get("file") or cfg.get("hf_hub_id"))
print("parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

with torch.inference_mode():
    probe = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    with torch.amp.autocast(device_type="cuda", enabled=True):
        sample = model(probe)
if isinstance(sample, (tuple, list)):
    sample = sample[0]
print("embedding shape:", tuple(sample.shape))
if sample.shape[-1] != 768:
    raise RuntimeError(f"expected a 768-d DINOv2 embedding, got {tuple(sample.shape)}")
del probe, sample
torch.cuda.empty_cache()


## 5. Embed the dataset

Embeddings are written to `outputs/dinov2_views_<size>.npz`. A later run reuses
that file when the path list matches, so the plots can be rebuilt without another
full pass. Mixed precision is on for the forward pass only.


In [ ]:
def _forward_view(file_paths, view: str, hue_shift: int, batch_size: int) -> np.ndarray:
    dataset = SareeViewDataset(file_paths, view, IMG_SIZE, hue_shift)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
    chunks = []
    for batch in tqdm(loader, desc=f"{view} hue={hue_shift} bs={batch_size}", leave=False):
        batch = batch.to(DEVICE, non_blocking=True)
        with torch.inference_mode(), torch.amp.autocast(device_type="cuda", enabled=True):
            features = model(batch)
        if isinstance(features, (tuple, list)):
            features = features[0]
        features = torch.nn.functional.normalize(features.float(), dim=-1)
        chunks.append(features.cpu())
    return torch.cat(chunks, dim=0).numpy()


def embed_view(file_paths, view: str, hue_shift: int = 0) -> np.ndarray:
    # Extract one view, halving the batch if the GPU runs out of memory.
    batch_size = BATCH_SIZE
    while True:
        try:
            return _forward_view(file_paths, view, hue_shift, batch_size)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if batch_size == 1:
                raise RuntimeError(
                    f"Out of memory at batch size 1 and image size {IMG_SIZE}. "
                    "Set IMG_SIZE = 224 in the config cell and rerun from that cell."
                ) from None
            batch_size = max(1, batch_size // 2)
            print(f"out of memory on {view}; retrying with batch size {batch_size}")


def early_fuse(view_matrix: dict[str, np.ndarray]) -> np.ndarray:
    total = np.zeros_like(next(iter(view_matrix.values())), dtype=np.float64)
    for name in VIEWS:
        block = view_matrix[name].astype(np.float64)
        total += block / np.clip(np.linalg.norm(block, axis=1, keepdims=True), 1e-12, None)
    return total / np.clip(np.linalg.norm(total, axis=1, keepdims=True), 1e-12, None)


def load_cached_views(file_paths):
    if FORCE_RECOMPUTE or not CACHE_PATH.exists():
        return None
    saved = np.load(CACHE_PATH, allow_pickle=False)
    current = np.array([str(path) for path in file_paths])
    if saved["paths"].shape != current.shape or not np.array_equal(saved["paths"], current):
        print("cache path list does not match this dataset; recomputing")
        return None
    print("loaded", CACHE_PATH)
    return {name: saved[name] for name in VIEWS}


view_emb = load_cached_views(paths)
if view_emb is None:
    view_emb = {}
    for view_name in VIEWS:
        view_emb[view_name] = embed_view(paths, view_name, hue_shift=0)
        torch.cuda.empty_cache()
    np.savez(
        CACHE_PATH,
        paths=np.array([str(path) for path in paths]),
        **view_emb,
    )
    print("saved", CACHE_PATH)

fused = early_fuse(view_emb)
print("fused", fused.shape)


## 6. Identification: Rank-1 and mAP

For every source photo that has two or more `.rf.` copies, one copy is the gallery
reference and the others are queries. The gallery therefore holds one image of each
such photo. A query's only relevant item is its sibling. With a single relevant
item, average precision is `1 / rank`, and mAP is the mean of that value. Rank-1 is
the fraction of queries whose sibling is the first result.

k-reciprocal re-ranking follows Zhong et al., CVPR 2017, with `k1=20`, `k2=6`,
and `lambda=0.3`. It is computed on the union of these queries and gallery images.


In [ ]:
def score_descriptor(name, embeddings):
    cosine_scores = cosine_query_scores(embeddings, query_index, gallery_index)
    rerank_scores = reranked_query_scores(embeddings, query_index, gallery_index, K1, K2, LAMBDA)
    return {
        "name": name,
        "cosine_scores": cosine_scores,
        "rerank_scores": rerank_scores,
        "cosine": rank1_and_map(cosine_scores, query_groups, gallery_groups),
        "rerank": rank1_and_map(rerank_scores, query_groups, gallery_groups),
    }


def pairwise_cosine(embeddings: np.ndarray) -> np.ndarray:
    features = embeddings.astype(np.float64)
    features = features / np.clip(np.linalg.norm(features, axis=1, keepdims=True), 1e-12, None)
    return features @ features.T


results = [score_descriptor(name, view_emb[name]) for name in VIEWS]
results.append(score_descriptor("early fusion", fused))

late_similarity = sum(pairwise_cosine(view_emb[name]) for name in VIEWS) / len(VIEWS)
late_cosine = late_similarity[np.ix_(query_index, gallery_index)]
late_rerank = reranked_from_similarity(late_similarity, query_index, gallery_index, K1, K2, LAMBDA)
results.append({
    "name": "late fusion",
    "cosine_scores": late_cosine,
    "rerank_scores": late_rerank,
    "cosine": rank1_and_map(late_cosine, query_groups, gallery_groups),
    "rerank": rank1_and_map(late_rerank, query_groups, gallery_groups),
})

header = f"{'descriptor':<16} {'cos Rank-1':>10} {'cos mAP':>10} {'rr Rank-1':>10} {'rr mAP':>10} {'queries':>8}"
print(header)
print("-" * len(header))
for row in results:
    print(
        f"{row['name']:<16} "
        f"{row['cosine']['rank1']:>10.3f} {row['cosine']['mAP']:>10.3f} "
        f"{row['rerank']['rank1']:>10.3f} {row['rerank']['mAP']:>10.3f} "
        f"{row['cosine']['n_queries']:>8d}"
    )

primary = next(row for row in results if row["name"] == "early fusion")


In [ ]:
labels = [row["name"] for row in results]
x = np.arange(len(labels))
width = 0.18
figure, axis = plt.subplots(figsize=(10, 4.2))
series = [
    ("cosine Rank-1", [row["cosine"]["rank1"] for row in results], "#0072B2"),
    ("cosine mAP", [row["cosine"]["mAP"] for row in results], "#56B4E9"),
    ("re-rank Rank-1", [row["rerank"]["rank1"] for row in results], "#E69F00"),
    ("re-rank mAP", [row["rerank"]["mAP"] for row in results], "#D55E00"),
]
for offset, (label, values, color) in enumerate(series):
    axis.bar(x + (offset - 1.5) * width, values, width, label=label, color=color)
axis.set_xticks(x)
axis.set_xticklabels(labels, rotation=15)
axis.set_ylim(0, 1)
axis.set_ylabel("score")
axis.set_title("Same-photo retrieval on Roboflow .rf. copies")
axis.legend(ncols=2, fontsize=8)
figure.tight_layout()
figure.savefig(OUTPUT_DIR / "identification_metrics.png", dpi=120)
plt.show()


## 7. What the matches look like

The grid uses early fusion plus k-reciprocal re-ranking. A green title is the
sibling copy. A red title is a different source photo. Weave names are printed
so the photo is identifiable. They are not the thing being scored.


In [ ]:
def positive_rank(scores: np.ndarray) -> np.ndarray:
    order = np.argsort(-scores, axis=1, kind="mergesort")
    ranks = np.empty(scores.shape[0], dtype=np.int32)
    for row, ranked in enumerate(order):
        target = query_groups[row]
        for place, col in enumerate(ranked, start=1):
            if gallery_groups[int(col)] == target:
                ranks[row] = place
                break
    return ranks


def show_matches(row_positions, scores, title, filename):
    figure, axes = plt.subplots(len(row_positions), 1 + TOPK, figsize=(2.1 * (1 + TOPK), 2.3 * len(row_positions)))
    if len(row_positions) == 1:
        axes = np.expand_dims(axes, 0)
    order = np.argsort(-scores, axis=1, kind="mergesort")
    for axis_row, query_row in enumerate(row_positions):
        query_rec = records[query_index[query_row]]
        panels = [(query_rec, "query")]
        for col in order[query_row, :TOPK]:
            panels.append((records[gallery_index[int(col)]], gallery_groups[int(col)] == query_groups[query_row]))
        for axis_col, (rec, flag) in enumerate(panels):
            with Image.open(rec["path"]) as handle:
                axes[axis_row, axis_col].imshow(handle.convert("RGB"))
            axes[axis_row, axis_col].axis("off")
            if axis_col == 0:
                axes[axis_row, axis_col].set_title(f"query ({query_rec['weave']})", fontsize=8, color="black")
            else:
                color = "#009E73" if flag else "#D55E00"
                axes[axis_row, axis_col].set_title("sibling" if flag else "other", fontsize=8, color=color)
    figure.suptitle(title)
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / filename, dpi=110)
    plt.show()


ranks = positive_rank(primary["rerank_scores"])
successes = list(np.flatnonzero(ranks == 1)[:3])
failures = list(np.flatnonzero(ranks > 1)[:3])
chosen = successes + failures
show_matches(chosen, primary["rerank_scores"], "Early fusion, k-reciprocal top-5", "retrieval_grid.png")
print(f"Rank-1 successes in this grid: {len(successes)}; misses: {len(failures)}")
print(f"median sibling rank: {np.median(ranks):.0f}; worst sibling rank: {ranks.max()}")


## 8. Hue-shift probe

The gallery stays on the original pixels. Each query is rotated by 90 OpenCV hue
units (about half the color wheel) before its views are built, then matched to
the original gallery. Grayscale and Canny of a recolored image should move less
than RGB. This is a stress test of color invariance, not a second labeled dataset.


In [ ]:
query_paths = [records[i]["path"] for i in query_index]
gallery_rows = gallery_index

hue_views = {}
for view_name in VIEWS:
    hue_views[view_name] = embed_view(query_paths, view_name, hue_shift=HUE_SHIFT)
    torch.cuda.empty_cache()
hue_fused = early_fuse(hue_views)


def between(query_emb, gallery_emb):
    query_emb = query_emb.astype(np.float64)
    gallery_emb = gallery_emb.astype(np.float64)
    query_emb = query_emb / np.clip(np.linalg.norm(query_emb, axis=1, keepdims=True), 1e-12, None)
    gallery_emb = gallery_emb / np.clip(np.linalg.norm(gallery_emb, axis=1, keepdims=True), 1e-12, None)
    return query_emb @ gallery_emb.T


hue_rows = []
for view_name in list(VIEWS) + ["early fusion"]:
    if view_name == "early fusion":
        scores = between(hue_fused, fused[gallery_rows])
    else:
        scores = between(hue_views[view_name], view_emb[view_name][gallery_rows])
    metrics = rank1_and_map(scores, query_groups, gallery_groups)
    original = next(row["cosine"]["rank1"] for row in results if row["name"] == view_name)
    hue_rows.append((view_name, original, metrics["rank1"], metrics["mAP"]))

print(f"{'descriptor':<16} {'original Rank-1':>16} {'hue Rank-1':>12} {'hue mAP':>10}")
for name, original, hue_rank, hue_map in hue_rows:
    print(f"{name:<16} {original:>16.3f} {hue_rank:>12.3f} {hue_map:>10.3f}")

figure, axis = plt.subplots(figsize=(8, 4))
names = [row[0] for row in hue_rows]
axis.bar(np.arange(len(names)) - 0.15, [row[1] for row in hue_rows], 0.3, label="original query", color="#0072B2")
axis.bar(np.arange(len(names)) + 0.15, [row[2] for row in hue_rows], 0.3, label="hue-shifted query", color="#E69F00")
axis.set_xticks(np.arange(len(names)))
axis.set_xticklabels(names, rotation=15)
axis.set_ylim(0, 1)
axis.set_ylabel("Rank-1")
axis.set_title("Sibling retrieval after a palette flip of the query")
axis.legend()
figure.tight_layout()
figure.savefig(OUTPUT_DIR / "hue_shift_rank1.png", dpi=120)
plt.show()


## 9. Verification

A pair is positive when both images are `.rf.` copies of one source photo, and
negative when they come from different source photos. The decision threshold is
chosen on one half of the source photos and reported on the other half, so the
threshold is not fit on the pairs it is judged on. The score is early-fusion cosine.


In [ ]:
from sklearn.metrics import auc, roc_curve

rng = np.random.default_rng(SEED)
usable_groups = sorted({records[i]["group"] for i in query_index})
order = rng.permutation(len(usable_groups))
cut = len(usable_groups) // 2
tune_groups = {usable_groups[i] for i in order[:cut]}
report_groups = {usable_groups[i] for i in order[cut:]}

members = {}
for index, record in enumerate(records):
    members.setdefault(record["group"], []).append(index)


def sample_pairs(group_set):
    positives = []
    pool = []
    for group in group_set:
        indexes = members[group]
        pool.extend(indexes)
        for left in range(len(indexes)):
            for right in range(left + 1, len(indexes)):
                positives.append((indexes[left], indexes[right]))
    positives = np.array(positives, dtype=np.int64)
    negatives = []
    guard = 0
    while len(negatives) < len(positives) and guard < len(positives) * 20:
        guard += 1
        left, right = rng.choice(pool, size=2, replace=False)
        if records[int(left)]["group"] == records[int(right)]["group"]:
            continue
        negatives.append((int(left), int(right)))
    negatives = np.array(negatives, dtype=np.int64)
    return positives, negatives


def pair_cosine(pairs):
    left = fused[pairs[:, 0]]
    right = fused[pairs[:, 1]]
    return np.sum(left * right, axis=1)


def choose_threshold(scores, labels):
    # Labels are 1 for the same source photo. Try every midpoint, plus the two
    # extremes that label every pair positive or every pair negative.
    order = np.argsort(scores)
    ordered_scores = scores[order]
    ordered_labels = labels[order]
    candidates = [float(ordered_scores[0] - 1e-6), float(ordered_scores[-1] + 1e-6)]
    for split in range(1, len(ordered_scores)):
        if ordered_scores[split] != ordered_scores[split - 1]:
            candidates.append(float(0.5 * (ordered_scores[split] + ordered_scores[split - 1])))
    best_threshold, best_accuracy = candidates[0], -1.0
    for candidate in candidates:
        accuracy = float(np.mean((ordered_scores >= candidate) == ordered_labels))
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_threshold = candidate
    return best_threshold, best_accuracy


tune_pos, tune_neg = sample_pairs(tune_groups)
report_pos, report_neg = sample_pairs(report_groups)
tune_scores = np.concatenate([pair_cosine(tune_pos), pair_cosine(tune_neg)])
tune_labels = np.concatenate([np.ones(len(tune_pos)), np.zeros(len(tune_neg))])
report_scores = np.concatenate([pair_cosine(report_pos), pair_cosine(report_neg)])
report_labels = np.concatenate([np.ones(len(report_pos)), np.zeros(len(report_neg))])

threshold, tune_accuracy = choose_threshold(tune_scores, tune_labels)
report_predictions = report_scores >= threshold
report_accuracy = float(np.mean(report_predictions == report_labels))
false_accepts = np.logical_and(report_predictions == 1, report_labels == 0).sum()
negatives_n = max(1, int((report_labels == 0).sum()))
positives_n = max(1, int((report_labels == 1).sum()))
true_accepts = np.logical_and(report_predictions == 1, report_labels == 1).sum()
print(f"threshold fit on {len(tune_groups)} source photos: {threshold:.3f} (tune accuracy {tune_accuracy:.3f})")
print(f"held-out photos: {len(report_groups)}")
print(f"held-out verification accuracy: {report_accuracy:.3f}")
print(f"held-out TAR: {true_accepts / positives_n:.3f} at FAR {false_accepts / negatives_n:.3f}")

fpr, tpr, _ = roc_curve(report_labels, report_scores)
figure, axis = plt.subplots(figsize=(5.2, 4.2))
axis.plot(fpr, tpr, color="#0072B2", label=f"AUC {auc(fpr, tpr):.3f}")
axis.plot([0, 1], [0, 1], color="0.7", linestyle="--")
axis.set_xlabel("false accept rate")
axis.set_ylabel("true accept rate")
axis.set_title("Same-photo verification, held-out source photos")
axis.legend()
figure.tight_layout()
figure.savefig(OUTPUT_DIR / "verification_roc.png", dpi=120)
plt.show()

figure, axis = plt.subplots(figsize=(6.5, 4))
axis.hist(report_scores[report_labels == 1], bins=40, alpha=0.75, label="same photo", color="#0072B2")
axis.hist(report_scores[report_labels == 0], bins=40, alpha=0.75, label="different photo", color="#E69F00")
axis.axvline(threshold, color="black", linestyle="--", label=f"threshold {threshold:.2f}")
axis.set_xlabel("early-fusion cosine")
axis.set_ylabel("pairs")
axis.set_title("Held-out pair scores")
axis.legend()
figure.tight_layout()
figure.savefig(OUTPUT_DIR / "verification_hist.png", dpi=120)
plt.show()


## 10. Inference

`match_topk` embeds queries and gallery images with the same three views, fuses
them, builds the cosine matrix, optionally applies k-reciprocal re-ranking on the
union of those images, and returns gallery indices. Paths that were already
embedded earlier in this session are reused.


In [ ]:
# (path, view, hue_shift) -> 768-d vector, filled from the dataset pass above.
_EMBED_CACHE = {}
for view_name in VIEWS:
    for index, path in enumerate(paths):
        _EMBED_CACHE[(str(path), view_name, 0)] = view_emb[view_name][index]


def embed_paths(file_paths, view: str, hue_shift: int = 0) -> np.ndarray:
    file_paths = [Path(path) for path in file_paths]
    keys = [(str(path.resolve()), view, hue_shift) for path in file_paths]
    missing = [i for i, key in enumerate(keys) if key not in _EMBED_CACHE]
    if missing:
        computed = embed_view([file_paths[i] for i in missing], view, hue_shift)
        for row, index in enumerate(missing):
            _EMBED_CACHE[keys[index]] = computed[row]
    return np.stack([_EMBED_CACHE[key] for key in keys], axis=0)


def fuse_paths(file_paths, hue_shift: int = 0) -> np.ndarray:
    view_matrix = {view_name: embed_paths(file_paths, view_name, hue_shift) for view_name in VIEWS}
    return early_fuse(view_matrix).astype(np.float64)


def match_topk(query_paths, gallery_paths, k: int = 5, use_rerank: bool = True):
    # Return top-k gallery indices and scores for each query.
    # Scores are higher for a closer match. With use_rerank, scores are the
    # negative k-reciprocal distance on the union of the query and gallery
    # images. Indices index gallery_paths.
    query_features = fuse_paths(query_paths)
    gallery_features = fuse_paths(gallery_paths)
    if use_rerank:
        union = np.concatenate([gallery_features, query_features], axis=0)
        similarity = union @ union.T
        distance = k_reciprocal_distance(similarity, k1=K1, k2=K2, lam=LAMBDA)
        n_gallery = len(gallery_paths)
        scores = -distance[n_gallery:, :n_gallery]
    else:
        scores = query_features @ gallery_features.T
    indices, top_scores = topk_indices(scores, k)
    return indices, top_scores


demo_rows = successes[:2] or list(range(min(2, len(query_index))))
demo_queries = [records[query_index[i]]["path"] for i in demo_rows]
demo_gallery = [records[i]["path"] for i in gallery_index]
demo_index, demo_scores = match_topk(demo_queries, demo_gallery, k=TOPK, use_rerank=True)
print("match_topk indices:")
print(demo_index)
print("match_topk scores:")
print(np.round(demo_scores, 3))


## 11. Unlabeled handloom images

These files have no design id, so they are not scored. The grid shows nearest
neighbors inside the handloom folder under the same fused descriptor, which is a
visual check that the embedding groups similar motifs.


In [ ]:
if HANDLOOM_ROOT is None:
    print("Handloom folder not found. Set HANDLOOM_ROOT to show nearest neighbors.")
else:
    handloom = discover_flat(HANDLOOM_ROOT)
    print("handloom images", len(handloom))
    hand_paths = [rec["path"] for rec in handloom]
    hand_features = fuse_paths(hand_paths)
    similarity = hand_features @ hand_features.T
    np.fill_diagonal(similarity, -np.inf)
    preview = min(4, len(handloom))
    figure, axes = plt.subplots(preview, 4, figsize=(8.5, 2.2 * preview))
    for row in range(preview):
        neighbor_cols = np.argsort(-similarity[row])[:3]
        columns = [row, *neighbor_cols.tolist()]
        for col, image_index in enumerate(columns):
            with Image.open(hand_paths[image_index]) as handle:
                axes[row, col].imshow(handle.convert("RGB"))
            axes[row, col].axis("off")
            axes[row, col].set_title("query" if col == 0 else f"{similarity[row, image_index]:.2f}", fontsize=8)
    figure.suptitle("Handloom nearest neighbors (no design labels, not scored)")
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / "handloom_neighbors.png", dpi=110)
    plt.show()


## 12. Efficiency

Parameter count is the frozen ViT-B/14. Latency is a single 3-channel forward
on the active GPU, so a full query (RGB, grayscale, and Canny) costs about three
of those forwards plus the edge extraction on CPU.


In [ ]:
import time

parameter_count = sum(parameter.numel() for parameter in model.parameters())
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
with torch.inference_mode():
    for _ in range(5):
        with torch.amp.autocast(device_type="cuda", enabled=True):
            model(dummy)
    torch.cuda.synchronize()
    start = time.perf_counter()
    repeats = 20
    for _ in range(repeats):
        with torch.amp.autocast(device_type="cuda", enabled=True):
            model(dummy)
    torch.cuda.synchronize()
elapsed_ms = (time.perf_counter() - start) * 1000 / repeats
del dummy
torch.cuda.empty_cache()

print(f"parameters: {parameter_count:,}")
print(f"embedding size: 768")
print(f"single-view GPU latency: {elapsed_ms:.1f} ms at {IMG_SIZE}px on {GPU_NAME}")
print(f"three-view query, GPU only: {3 * elapsed_ms:.1f} ms")

import json

payload = {
    "approach_note": APPROACH,
    "gpu": GPU_NAME,
    "image_size": IMG_SIZE,
    "parameters": parameter_count,
    "embedding_dim": 768,
    "single_view_latency_ms": elapsed_ms,
    "identification": {
        row["name"]: {"cosine": row["cosine"], "rerank": row["rerank"]}
        for row in results
    },
    "hue_shift": [
        {"descriptor": name, "original_rank1": original, "hue_rank1": hue_rank, "hue_mAP": hue_map}
        for name, original, hue_rank, hue_map in hue_rows
    ],
    "verification": {
        "threshold": threshold,
        "held_out_accuracy": report_accuracy,
        "held_out_tar": true_accepts / positives_n,
        "held_out_far": false_accepts / negatives_n,
        "auc": float(auc(fpr, tpr)),
    },
}
metrics_path = OUTPUT_DIR / "metrics.json"
metrics_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("wrote", metrics_path)
print("approach note characters:", len(APPROACH))


## 13. Disclosure

- Backbone: public DINOv2 ViT-B/14 via `timm.create_model("vit_base_patch14_dinov2", pretrained=True, num_classes=0)`.
- No fine-tuning, no ArcFace, no use of weave names as class ids.
- Kaggle / Roboflow Indian fabric-pattern export, MIT license, plus the local unlabeled handloom folder. The handloom images are not redistributed in this repository.
- Re-ranking: k-reciprocal encoding as described by Zhong, Zheng, Cao, and Li, CVPR 2017. `k1=20`, `k2=6`, `lambda=0.3`.
- Canny thresholds 100 and 200 were fixed before looking at scores.
